# Evaluación de modelos

Validación del entrenamiento de los 6 modelos del torneo y de la coherencia de sus salidas:

- Se reconstruye la mejor configuración de cada modelo a partir de los resultados exportados por el notebook de entrenamiento.
- Se verifican sus métricas en el set de validación (ajustados con el 80% de entrenamiento) y se comparan contra un modelo base.
- Cada modelo se reentrena con el 100% de `train.csv`, igual que el ganador, y predice los datos crudos de `test.csv`, para comparar la forma de su salida con las predicciones generadas por la inferencia.
- Los modelos que no generan predicciones coherentes se marcan como **descartados**.

Los resultados se exportan a `reports/evaluacion_modelos.json`, que alimenta el artefacto de visualización.

## Librerías

In [ ]:
import json
import os
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from scipy.stats import ks_2samp
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

## Configuración

Rutas, parámetros leídos de `pipeline_config.yaml` y criterios de coherencia. Un modelo se descarta si cumple **cualquiera** de estos criterios:

| Criterio | Regla |
|---|---|
| Predicciones inválidas | Alguna predicción de test es NaN, infinita o ≤ 0 |
| Fuera del rango de train | Alguna predicción de test < 0.5 × mínimo o > 1.5 × máximo de `SalePrice` en train |
| Distribución distinta | Estadístico de Kolmogorov-Smirnov entre las predicciones de test y `SalePrice` real de train > 0.1 |
| Peor que un modelo base | RMSE real en validación ≥ el de predecir siempre la mediana de `SalePrice` de entrenamiento |

In [ ]:
# Rutas: modificar estas variables si cambia la ubicación de las fuentes o de las salidas
PROJ_DIR = Path(os.getenv("PROJ_DIR", ".."))
CONFIG_FILE = Path(os.getenv("PIPELINE_CONFIG", PROJ_DIR / "pipeline_config.yaml"))
RAW_DATA_DIR = Path(os.getenv("RAW_DATA_DIR", PROJ_DIR / "data" / "raw"))
REPORTS_DIR = Path(os.getenv("REPORTS_DIR", PROJ_DIR / "reports"))

TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
EVALUACION_FILE = "evaluacion_modelos.json"

with open(CONFIG_FILE, encoding="utf-8") as archivo:
    config = yaml.safe_load(archivo)

MODEL_FEATURES = config["MODEL_FEATURES"]
TARGET = config["TARGET"]
ID_COL = config["ID_COLUMN"]
TEST_SIZE = config["SPLIT"]["test_size"]
RANDOM_STATE = config["SPLIT"]["random_state"]

# Criterios de coherencia
FACTOR_MINIMO = 0.5
FACTOR_MAXIMO = 1.5
UMBRAL_KS = 0.1

CLASES_MODELOS = {
    "Ridge": Ridge,
    "Lasso": Lasso,
    "ElasticNet": ElasticNet,
    "RandomForestRegressor": RandomForestRegressor,
    "GradientBoostingRegressor": GradientBoostingRegressor,
    "MLPRegressor": MLPRegressor,
}
TIPOS_MODELOS = {
    "Ridge": "Regresión",
    "Lasso": "Regresión",
    "ElasticNet": "Regresión",
    "RandomForest": "Ensamble",
    "GradientBoosting": "Ensamble",
    "MLPRegressor": "Red neuronal",
}

## Carga de datos

Datos crudos de train y test, la misma división 80/20 del entrenamiento, los resultados del torneo y las predicciones de test generadas por la inferencia.

In [ ]:
train = pd.read_csv(RAW_DATA_DIR / TRAIN_FILE, index_col=ID_COL)
test = pd.read_csv(RAW_DATA_DIR / TEST_FILE, index_col=ID_COL)

X = train[MODEL_FEATURES]
y = np.log1p(train[TARGET])
X_test = test[MODEL_FEATURES]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
y_train_real = np.expm1(y_train)
y_val_real = np.expm1(y_val)

torneo_cv = pd.read_csv(PROJ_DIR / config["TOURNAMENT"]["cv"])
torneo_validacion = pd.read_csv(PROJ_DIR / config["TOURNAMENT"]["validacion"])
with open(PROJ_DIR / config["TOURNAMENT"]["mejores_modelos"], encoding="utf-8") as archivo:
    mejores_modelos = yaml.safe_load(archivo)

predicciones_inferencia = pd.read_csv(
    PROJ_DIR / config["INFERENCE"]["predicciones_test"], index_col=ID_COL
)[TARGET]

print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}")
print(f"Modelos del torneo: {list(mejores_modelos)}")
print(f"Predicciones de la inferencia: {len(predicciones_inferencia)}")

## Reconstrucción de los modelos

Cada modelo se arma como `feature engineering + modelo`, con el pipeline de FE clonado sin ajustar y la clase y parámetros de su mejor configuración.

In [ ]:
pipeline_fe = clone(joblib.load(PROJ_DIR / config["OUTPUTS"]["pipeline"]))


def construir_modelo(definicion):
    """Pipeline sin ajustar: feature engineering + modelo con los parámetros dados."""
    parametros = dict(definicion["parametros"])
    if "hidden_layer_sizes" in parametros:
        parametros["hidden_layer_sizes"] = tuple(parametros["hidden_layer_sizes"])
    modelo = CLASES_MODELOS[definicion["clase"]](**parametros)
    return Pipeline(steps=[("fe", clone(pipeline_fe)), ("modelo", modelo)])


modelos = {nombre: construir_modelo(definicion) for nombre, definicion in mejores_modelos.items()}

pd.DataFrame(
    {
        "modelo": list(mejores_modelos),
        "clase": [definicion["clase"] for definicion in mejores_modelos.values()],
        "tipo": [TIPOS_MODELOS[nombre] for nombre in mejores_modelos],
    }
)

## Métricas en validación

Cada modelo se ajusta con el 80% de entrenamiento y predice el set de validación. Se verifica que el RMSE coincida con el del torneo y se calcula el modelo base: predecir siempre la mediana de `SalePrice` de entrenamiento.

In [ ]:
rango_val = y_val_real.max() - y_val_real.min()
rmse_base = root_mean_squared_error(y_val_real, np.full(len(y_val_real), y_train_real.median()))

predicciones_val = {}
for nombre, modelo in modelos.items():
    predicciones_val[nombre] = np.expm1(clone(modelo).fit(X_train, y_train).predict(X_val))

verificacion = torneo_validacion.set_index("modelo")[["rmse_real_val"]].assign(
    rmse_recalculado=pd.Series(
        {nombre: root_mean_squared_error(y_val_real, pred) for nombre, pred in predicciones_val.items()}
    )
)
verificacion["coincide"] = np.isclose(verificacion["rmse_real_val"], verificacion["rmse_recalculado"])

print(f"RMSE real del modelo base (mediana): {rmse_base:,.0f}")
verificacion.round(0)

## Predicciones de test con los modelos reentrenados

Cada modelo se reentrena con el 100% de `train.csv`, igual que el ganador de la inferencia, y predice los datos crudos de `test.csv`. Se verifica que el modelo ganador reproduzca las predicciones de la inferencia.

In [ ]:
predicciones_test = {}
for nombre, modelo in modelos.items():
    predicciones_test[nombre] = np.expm1(clone(modelo).fit(X, y).predict(X_test))

ganador = config["BEST_MODEL"]["nombre"]
print(f"El ganador ({ganador}) reproduce la inferencia: "
      f"{np.allclose(predicciones_test[ganador], predicciones_inferencia.loc[X_test.index])}")

pd.DataFrame(predicciones_test, index=X_test.index).describe().round(0)

## Coherencia de las salidas

Se aplican los 4 criterios a cada modelo. El estadístico KS contra las predicciones de la inferencia se reporta solo como referencia, no como criterio de descarte.

In [ ]:
minimo_permitido = FACTOR_MINIMO * train[TARGET].min()
maximo_permitido = FACTOR_MAXIMO * train[TARGET].max()

filas = []
for nombre, pred in predicciones_test.items():
    rmse_real_val = torneo_validacion.set_index("modelo").loc[nombre, "rmse_real_val"]
    invalidas = int((~np.isfinite(pred) | (pred <= 0)).sum())
    fuera_rango = int(((pred < minimo_permitido) | (pred > maximo_permitido)).sum())
    ks_train = ks_2samp(pred, train[TARGET]).statistic
    ks_inferencia = ks_2samp(pred, predicciones_inferencia).statistic

    motivos = []
    if invalidas:
        motivos.append(f"{invalidas} predicciones inválidas")
    if fuera_rango:
        motivos.append(f"{fuera_rango} predicciones fuera del rango de train")
    if ks_train > UMBRAL_KS:
        motivos.append(f"distribución distinta a train (KS = {ks_train:.3f})")
    if rmse_real_val >= rmse_base:
        motivos.append("RMSE en validación peor que el modelo base")

    filas.append(
        {
            "modelo": nombre,
            "invalidas": invalidas,
            "fuera_rango": fuera_rango,
            "ks_train": ks_train,
            "ks_inferencia": ks_inferencia,
            "peor_que_base": bool(rmse_real_val >= rmse_base),
            "descartado": bool(motivos),
            "motivos": "; ".join(motivos),
        }
    )

coherencia = pd.DataFrame(filas)
coherencia.round(3)

## Ranking final

Solo entran al ranking los modelos que no fueron descartados, ordenados por el RMSE real en validación.

In [ ]:
resumen = torneo_validacion.merge(coherencia, on="modelo")
ranking = resumen[~resumen["descartado"]].sort_values("rmse_real_val").reset_index(drop=True)
ranking.index = ranking.index + 1

ranking[["modelo", "rmse_real_val", "nrmse_rango_val", "rmse_log_cv", "ks_train"]].round(4)

## Exportar la evaluación

Se guarda en `reports/evaluacion_modelos.json` todo lo necesario para el artefacto: métricas, configuraciones de la validación cruzada, predicciones de validación y de test de cada modelo, distribución real de train, predicciones de la inferencia y criterios de coherencia.

In [ ]:
def redondear(valores, decimales=0):
    return [round(float(valor), decimales) for valor in valores]


evaluacion = {
    "generado": date.today().isoformat(),
    "target": TARGET,
    "n_train": len(X_train),
    "n_val": len(X_val),
    "n_test": len(X_test),
    "k_folds": config["BEST_MODEL"]["k_folds"],
    "ganador_inferencia": ganador,
    "rango_val": float(rango_val),
    "rmse_base": float(rmse_base),
    "criterios": {
        "factor_minimo": FACTOR_MINIMO,
        "factor_maximo": FACTOR_MAXIMO,
        "minimo_permitido": float(minimo_permitido),
        "maximo_permitido": float(maximo_permitido),
        "umbral_ks": UMBRAL_KS,
    },
    "train_real": redondear(train[TARGET]),
    "val_real": redondear(y_val_real),
    "inferencia_test": redondear(predicciones_inferencia),
    "modelos": [
        {
            "nombre": nombre,
            "tipo": TIPOS_MODELOS[nombre],
            "clase": mejores_modelos[nombre]["clase"],
            "metricas": {
                clave: float(valor)
                for clave, valor in fila.items()
                if clave.startswith(("rmse", "nrmse", "brecha"))
            },
            "coherencia": {
                "invalidas": int(fila["invalidas"]),
                "fuera_rango": int(fila["fuera_rango"]),
                "ks_train": float(fila["ks_train"]),
                "ks_inferencia": float(fila["ks_inferencia"]),
                "peor_que_base": bool(fila["peor_que_base"]),
            },
            "descartado": bool(fila["descartado"]),
            "motivos": fila["motivos"],
            "cv": torneo_cv[torneo_cv["modelo"] == nombre]
            .sort_values("ranking")
            .drop(columns="modelo")
            .to_dict(orient="records"),
            "pred_val": redondear(predicciones_val[nombre]),
            "pred_test": redondear(predicciones_test[nombre]),
        }
        for nombre, fila in resumen.set_index("modelo").iterrows()
    ],
}

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ruta_evaluacion = REPORTS_DIR / EVALUACION_FILE
with open(ruta_evaluacion, "w", encoding="utf-8") as archivo:
    json.dump(evaluacion, archivo, ensure_ascii=False)

print(f"Evaluación guardada en: {ruta_evaluacion.resolve()}")